In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, abs, avg
from pyspark.ml.feature import VectorAssembler, StringIndexer

spark = SparkSession.builder.appName("BigDataProject").getOrCreate()


In [ ]:
# Data read
df = spark.read.csv("application_data.csv", header=True, inferSchema=True)

In [ ]:
df.show()

+----------+------+------------------+-----------+------------+---------------+------------+----------------+----------+-----------+---------------+---------------+--------------------+--------------------+--------------------+-----------------+--------------------------+----------+-------------+-----------------+---------------+-----------+----------+--------------+---------------+----------------+----------+----------+---------------+---------------+--------------------+---------------------------+--------------------------+-----------------------+--------------------------+--------------------------+---------------------------+----------------------+----------------------+-----------------------+--------------------+-------------------+-------------------+-------------------+--------------+----------------+---------------------------+------------------+--------------+-------------+-------------+-------------+-------------+------------+--------------------+--------------+------------

In [ ]:
df.printSchema()

root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- TARGET: integer (nullable = true)
 |-- NAME_CONTRACT_TYPE: string (nullable = true)
 |-- CODE_GENDER: string (nullable = true)
 |-- FLAG_OWN_CAR: string (nullable = true)
 |-- FLAG_OWN_REALTY: string (nullable = true)
 |-- CNT_CHILDREN: integer (nullable = true)
 |-- AMT_INCOME_TOTAL: double (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- AMT_GOODS_PRICE: double (nullable = true)
 |-- NAME_TYPE_SUITE: string (nullable = true)
 |-- NAME_INCOME_TYPE: string (nullable = true)
 |-- NAME_EDUCATION_TYPE: string (nullable = true)
 |-- NAME_FAMILY_STATUS: string (nullable = true)
 |-- NAME_HOUSING_TYPE: string (nullable = true)
 |-- REGION_POPULATION_RELATIVE: double (nullable = true)
 |-- DAYS_BIRTH: integer (nullable = true)
 |-- DAYS_EMPLOYED: integer (nullable = true)
 |-- DAYS_REGISTRATION: double (nullable = true)
 |-- DAYS_ID_PUBLISH: integer (nullable = true)
 |-- OWN_CAR_AG

In [ ]:
#Data Cleaning

#Menghapus baris yang memiliki nilai null di kolom penting
df = df.dropna(subset=["SK_ID_CURR", "TARGET"])

#Mengisi nilai null pada kolom numerik dengan 0 atau rata-rata
df = df.fillna({"AMT_ANNUITY": 0})

In [ ]:
#Data Cleaning (2)
#check duplicate data
print(df.count())
print(df.dropDuplicates().count())

307511
307511


In [ ]:
#Data Cleaning(3)
#check datatype
df.printSchema()

root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- TARGET: integer (nullable = true)
 |-- NAME_CONTRACT_TYPE: string (nullable = true)
 |-- CODE_GENDER: string (nullable = true)
 |-- FLAG_OWN_CAR: string (nullable = true)
 |-- FLAG_OWN_REALTY: string (nullable = true)
 |-- CNT_CHILDREN: integer (nullable = true)
 |-- AMT_INCOME_TOTAL: double (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)
 |-- AMT_ANNUITY: double (nullable = false)
 |-- AMT_GOODS_PRICE: double (nullable = true)
 |-- NAME_TYPE_SUITE: string (nullable = true)
 |-- NAME_INCOME_TYPE: string (nullable = true)
 |-- NAME_EDUCATION_TYPE: string (nullable = true)
 |-- NAME_FAMILY_STATUS: string (nullable = true)
 |-- NAME_HOUSING_TYPE: string (nullable = true)
 |-- REGION_POPULATION_RELATIVE: double (nullable = true)
 |-- DAYS_BIRTH: integer (nullable = true)
 |-- DAYS_EMPLOYED: integer (nullable = true)
 |-- DAYS_REGISTRATION: double (nullable = true)
 |-- DAYS_ID_PUBLISH: integer (nullable = true)
 |-- OWN_CAR_A

In [ ]:
#Data Cleaning(4)
#check invalid values
df.filter(col("AMT_INCOME_TOTAL") <= 0).show()

#delete invalid values
df = df.filter(col("AMT_INCOME_TOTAL") > 0)

+----------+------+------------------+-----------+------------+---------------+------------+----------------+----------+-----------+---------------+---------------+----------------+-------------------+------------------+-----------------+--------------------------+----------+-------------+-----------------+---------------+-----------+----------+--------------+---------------+----------------+----------+----------+---------------+---------------+--------------------+---------------------------+--------------------------+-----------------------+--------------------------+--------------------------+---------------------------+----------------------+----------------------+-----------------------+-----------------+------------+------------+------------+--------------+----------------+---------------------------+---------------+--------------+-------------+-------------+-------------+-------------+------------+--------------------+--------------+-----------------------+-----------------+----

In [ ]:
#Data Transformation & Aggregation
from pyspark.sql.functions import col, abs, avg, count, when

#Mengubah nilai negatif pada DAYS_BIRTH menjadi umur positif (tahun)
df = df.withColumn("AGE", abs(col("DAYS_BIRTH")) / 365)

#Mengubah lama bekerja dari hari menjadi tahun kerja
df = df.withColumn(
    "EMPLOYED_YEARS",
    (abs(col("DAYS_EMPLOYED")) / 365).cast("int")
)

#Membuat rasio kredit terhadap total pendapatan
df = df.withColumn(
    "CREDIT_INCOME_RATIO",
    col("AMT_CREDIT") / col("AMT_INCOME_TOTAL")
)

#Rata-rata pinjaman berdasarkan tipe kontrak
df_agg = df.groupBy("NAME_CONTRACT_TYPE").agg(avg("AMT_CREDIT").alias("AVG_CREDIT"))
df_agg.show()

#Menghitung jumlah applicant dan rata-rata income berdasarkan TARGET
df_agg_target = df.groupBy("TARGET").agg(
    count("*").alias("TOTAL_APPLICANTS"),
    avg("AMT_INCOME_TOTAL").alias("AVG_INCOME"),
    avg("AMT_CREDIT").alias("AVG_CREDIT")
)

df_agg_target.show()

+------------------+-----------------+
|NAME_CONTRACT_TYPE|       AVG_CREDIT|
+------------------+-----------------+
|   Revolving loans|324017.9821715223|
|        Cash loans|627965.7325379539|
+------------------+-----------------+

+------+----------------+------------------+-----------------+
|TARGET|TOTAL_APPLICANTS|        AVG_INCOME|       AVG_CREDIT|
+------+----------------+------------------+-----------------+
|     1|           24825|165611.76090634443| 557778.527673716|
|     0|          282686|169077.72226581795|602648.2820019386|
+------+----------------+------------------+-----------------+



In [ ]:
#Feature Engineering
from pyspark.ml.feature import StringIndexer, VectorAssembler

#Cek jika kolom GENDER_INDEX sudah ada, jika ya, hapus dulu
if "GENDER_INDEX" in df.columns:
    df = df.drop("GENDER_INDEX")

#Ubah kategori Gender menjadi index angka
indexer = StringIndexer(inputCol="CODE_GENDER", outputCol="GENDER_INDEX")
df = indexer.fit(df).transform(df)

#Gabung fitur pilihan ke dalam satu kolom 'features'
#Cek untuk kolom 'features' agar tidak bentrok
if "features" in df.columns:
    df = df.drop("features")

assembler = VectorAssembler(inputCols=["AGE", "AMT_INCOME_TOTAL", "GENDER_INDEX"], outputCol="features")
df_final = assembler.transform(df)

# Tampilkan hasil untuk memastika
df_final.select("CODE_GENDER", "GENDER_INDEX", "features").show(5, truncate=False)

+-----------+------------+---------------------------------+
|CODE_GENDER|GENDER_INDEX|features                         |
+-----------+------------+---------------------------------+
|M          |1.0         |[25.92054794520548,202500.0,1.0] |
|F          |0.0         |[45.93150684931507,270000.0,0.0] |
|M          |1.0         |[52.18082191780822,67500.0,1.0]  |
|F          |0.0         |[52.06849315068493,135000.0,0.0] |
|M          |1.0         |[54.608219178082194,121500.0,1.0]|
+-----------+------------+---------------------------------+
only showing top 5 rows


In [ ]:
df.write.csv("processed_dataset", header=True)